## New‑Enquiry Triage & Intake Agent for LegalTech Solicitors

Use the same agentic patterns as in openai/lab2 to handle inbound enquiries (“I need help with a funding round / contract / dispute”) instead of just outbound cold emails.

### Imports and setup

In [14]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from typing import Dict
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

load_dotenv(override=True)

True

### Tools: logging, booking link, sending reply email

In [15]:
@function_tool
def log_matter_intake(
    client_email: str,
    matter_type: str,
    urgency: str,
    summary: str,
) -> Dict[str, str]:
   
    log_path = Path("intake_log.jsonl")
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "client_email": client_email,
        "matter_type": matter_type,
        "urgency": urgency,
        "summary": summary,
    }
    with log_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")
    return {"status": "logged", "path": str(log_path)}

In [16]:
@function_tool
def get_booking_link(lawyer_name: str = "LegalTech Solicitors") -> Dict[str, str]:
  
    return {
        "lawyer_name": lawyer_name,
        "url": "https://calendly.com/sankari-s2009/30min"
    }

In [17]:
@function_tool
def send_reply_email(
    to_email: str,
    subject: str,
    body: str,
) -> Dict[str, str]:
   
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("sankari.s2009@gmail.com")  
    to_email_obj = To(to_email)
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email_obj, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "sent"}

### AGents

In [18]:
intake_classifier = Agent(
    name="Intake Classifier",
    instructions="""
You read an inbound enquiry email sent to LegalTech Solicitors.

Your job is to:
1) Identify the primary matter type (one of: commercial_contracts, employment, ip, disputes, data_protection, other).
2) Identify urgency (one of: low, medium, high, critical).
3) Briefly summarise the enquiry in 1–2 sentences.

Return your answer in strict JSON with keys: matter_type, urgency, summary.
""",
    model="gpt-4o-mini",
)

intake_classifier_tool = intake_classifier.as_tool(
    tool_name="intake_classifier",
    tool_description="Classify an inbound enquiry and summarise it as JSON.",
)

In [19]:
matter_classifier = Agent(
    name="Matter Classifier",
    instructions="""
You help classify new client matters for LegalTech Solicitors.

You receive:
- The original enquiry email text.
- A JSON classification with matter_type, urgency, and summary.

Write a clear, professional reply email that:
- Acknowledges the enquiry.
- Reflects their matter type and urgency.
- Asks 3–6 specific clarifying questions needed to scope the work.
- Sets expectations for next steps.
Offer a call if appropriate, but do not invent a booking link yourself; you will be given one.

Do not use placeholders like [CLIENT NAME]; write so the email can be sent as-is, addressing the sender as "you".
Return only the email body as plain text.

At the end of the email, always add this signature:
Best regards,
Siva Sankari
LegalTech Solicitors
""",
    model="gpt-4o-mini",
)

matter_classifier_tool = matter_classifier.as_tool(
    tool_name="matter_classifier",
    tool_description="Draft a reply email to clarify and scope the matter.",
)

### Orchestrator

In [20]:
tools = [
    intake_classifier_tool,
    matter_classifier_tool,
    get_booking_link,
    log_matter_intake,
    send_reply_email,
]

intake_manager_instructions = """
You are the Intake Manager for LegalTech Solicitors.

You receive a message that always starts with a line:
`Client email: some.address@example.com`
followed by a blank line and then the full enquiry email text.

Your goals for each enquiry are:

1) Use the intake_classifier tool
   - Pass the full enquiry email text (not including the Client email line).
   - Read its JSON result (matter_type, urgency, summary).

2) Use the matter_classifier tool
   - Provide both the original enquiry text and the JSON classification (you can include the JSON directly in the input).
   - Get back a well-written reply email body.

3) Use the get_booking_link tool exactly once
   - Retrieve a booking URL.
   - Incorporate this URL naturally into the reply email (e.g. “you can book a call here: <url>”).

4) Use the log_matter_intake tool exactly once
   - client_email: the email address from the first line.
   - matter_type, urgency, summary: taken from the classifier’s JSON.

5) Use the send_reply_email tool exactly once
   - to_email: the client email.
   - subject: something like “Re: your enquiry with LegalTech Solicitors”.
   - body: the reply email text from the matter_classifier, updated to include the booking link.

Crucial rules:
- Always use intake_classifier and matter_scoper; do not perform their work yourself.
- Always call get_booking_link and log_matter_intake exactly once per enquiry.
- Call send_reply_email exactly once per enquiry.
- Be careful to reuse the JSON fields you get back; do not change their meaning.
"""

intake_manager = Agent(
    name="Intake Manager",
    instructions=intake_manager_instructions,
    tools=tools,
    model="gpt-4o-mini",
)

### Testing

In [21]:
client_email = "sankari.s2009@gmail.com"

enquiry_email = """
Hi LegalTech Solicitors,

We're a UK SaaS startup preparing for our Series A round in June.
We need help reviewing and updating our investment documents and existing customer contracts,
and making sure we're covered from a data protection perspective.

Timeline is quite tight and we'd like to speak to someone this week if possible.

Best,
Siva
"""

message = f"""Client email: {client_email}

{enquiry_email}
"""

with trace("LegalTech Solicitors Intake"):
    result = await Runner.run(intake_manager, message)

print("Final agent output (may just describe what it did):")
print(result.final_output)

Final agent output (may just describe what it did):
The enquiry from Siva Sankari has been successfully processed. Here’s a summary of the actions taken:

1. **Intake Classification**:
   - **Matter Type**: Commercial Contracts
   - **Urgency**: High
   - **Summary**: A UK SaaS startup is seeking assistance with reviewing and updating investment documents and customer contracts in preparation for their Series A round, focusing on compliance with data protection regulations.

2. **Reply Email Sent**:
   - I drafted a reply email that requests more details from Siva about the investment documents and customer contracts, ensuring clarity on their needs. 
   - Included a booking link for a call: [Book a Call](https://calendly.com/sankari-s2009/30min).

3. **Matter Logged**: The details of the matter have been logged successfully for future reference.

If there's anything else you need, let me know!
